# 🚦 End-to-End Transportation Pipeline with Databricks Lakeflow (Spark Declarative Pipelines)

**Author:** Tushar  
**Domain:** Transportation / Urban Mobility  
**Stack:** Databricks, Lakeflow, PySpark, Delta Lake, Unity Catalog  

---

## Project Overview

This notebook demonstrates an **end-to-end data pipeline** built using **Databricks Lakeflow Spark Declarative Pipelines** — the next-generation, declarative approach to building reliable, scalable ETL pipelines on Databricks.

### Architecture: Medallion (Bronze → Silver → Gold)

```
Raw Sources (CSV/JSON)
        │
        ▼
  [Bronze Layer]  ← Raw ingestion via Auto Loader
        │
        ▼
  [Silver Layer]  ← Cleaned, validated, joined
        │
        ▼
   [Gold Layer]   ← Business-level aggregations & KPIs
```

### Domain: Urban Transportation
- **Trips data** — ride records (start/end location, duration, fare)
- **Vehicles data** — fleet metadata (type, capacity, fuel type)
- **Routes data** — route definitions and zones
- **Events data** — real-time delay/incident events

---

## 📦 Section 1: Setup & Prerequisites

> **Note:** This pipeline is designed to run on a Databricks cluster with Lakeflow (Declarative Pipelines) enabled.  
> For local simulation, we use PySpark + Delta Lake.

In [ ]:
# ─── Install dependencies (if running locally / in a standard Databricks notebook) ───
# In a Lakeflow pipeline, these are pre-installed on the cluster.

# %pip install delta-spark pyspark faker  # Uncomment if running locally

import os
import json
import random
from datetime import datetime, timedelta

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, DoubleType,
    TimestampType, BooleanType
)

print("✅ Imports successful")

In [ ]:
# ─── Spark Session (auto-available in Databricks; explicit for local runs) ───

spark = (
    SparkSession.builder
    .appName("LakeflowTransportationPipeline")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print(f"✅ Spark version: {spark.version}")

In [ ]:
# ─── Config ───

BASE_PATH     = "/tmp/transport_lakehouse"      # Change to DBFS/ADLS path in Databricks
BRONZE_PATH   = f"{BASE_PATH}/bronze"
SILVER_PATH   = f"{BASE_PATH}/silver"
GOLD_PATH     = f"{BASE_PATH}/gold"
LANDING_PATH  = f"{BASE_PATH}/landing"          # Raw source files land here
CHECKPOINT    = f"{BASE_PATH}/checkpoints"

# Unity Catalog targets (used in Lakeflow pipeline definition)
CATALOG       = "main"
SCHEMA        = "transportation"

for path in [LANDING_PATH, BRONZE_PATH, SILVER_PATH, GOLD_PATH, CHECKPOINT]:
    os.makedirs(path, exist_ok=True)

print("✅ Paths configured")

---
## 🏭 Section 2: Synthetic Data Generation

Simulating raw source data that would arrive from operational systems (GPS trackers, dispatch systems, IoT sensors).

In [ ]:
# ─── Generate: Vehicles master data ───

random.seed(42)

VEHICLE_TYPES  = ["Bus", "Metro", "Tram", "Cab", "Electric Bus"]
FUEL_TYPES     = ["Diesel", "Electric", "CNG", "Hybrid"]
CITIES         = ["Delhi", "Mumbai", "Bangalore", "Chennai", "Pune"]

vehicles = [
    {
        "vehicle_id"   : f"VH{str(i).zfill(4)}",
        "vehicle_type" : random.choice(VEHICLE_TYPES),
        "capacity"     : random.choice([20, 40, 60, 80, 100]),
        "fuel_type"    : random.choice(FUEL_TYPES),
        "city"         : random.choice(CITIES),
        "manufacture_year": random.randint(2010, 2023),
        "is_active"    : random.choice([True, True, True, False])  # 75% active
    }
    for i in range(1, 201)
]

vehicles_df = spark.createDataFrame(vehicles)
vehicles_df.write.mode("overwrite").json(f"{LANDING_PATH}/vehicles")

print(f"✅ Generated {vehicles_df.count()} vehicle records")
vehicles_df.show(5, truncate=False)

In [ ]:
# ─── Generate: Routes master data ───

ZONES = ["North", "South", "East", "West", "Central"]

routes = [
    {
        "route_id"         : f"RT{str(i).zfill(3)}",
        "route_name"       : f"Route-{i}",
        "origin_zone"      : random.choice(ZONES),
        "destination_zone" : random.choice(ZONES),
        "distance_km"      : round(random.uniform(5.0, 80.0), 2),
        "expected_duration_min": random.randint(15, 120),
        "city"             : random.choice(CITIES)
    }
    for i in range(1, 51)
]

routes_df = spark.createDataFrame(routes)
routes_df.write.mode("overwrite").json(f"{LANDING_PATH}/routes")

print(f"✅ Generated {routes_df.count()} route records")
routes_df.show(5, truncate=False)

In [ ]:
# ─── Generate: Trips transactional data ───

STATUSES      = ["completed", "completed", "completed", "cancelled", "in_progress"]
PAYMENT_MODES = ["Cash", "UPI", "Card", "Pass"]

base_ts = datetime(2024, 1, 1)

trips = []
for i in range(1, 5001):
    trip_start = base_ts + timedelta(
        days=random.randint(0, 364),
        hours=random.randint(0, 23),
        minutes=random.randint(0, 59)
    )
    duration_min = random.randint(10, 130)
    trip_end     = trip_start + timedelta(minutes=duration_min)
    fare         = round(random.uniform(10.0, 500.0), 2)

    trips.append({
        "trip_id"         : f"TR{str(i).zfill(6)}",
        "vehicle_id"      : f"VH{str(random.randint(1, 200)).zfill(4)}",
        "route_id"        : f"RT{str(random.randint(1, 50)).zfill(3)}",
        "passenger_count" : random.randint(1, 80),
        "trip_start_ts"   : trip_start.isoformat(),
        "trip_end_ts"     : trip_end.isoformat(),
        "duration_min"    : duration_min,
        "fare_amount"     : fare,
        "payment_mode"    : random.choice(PAYMENT_MODES),
        "status"          : random.choice(STATUSES),
        "delay_min"       : random.choice([0, 0, 0, random.randint(1, 60)])  # mostly on-time
    })

trips_df = spark.createDataFrame(trips)
trips_df.write.mode("overwrite").json(f"{LANDING_PATH}/trips")

print(f"✅ Generated {trips_df.count()} trip records")
trips_df.show(5, truncate=False)

In [ ]:
# ─── Generate: Events / Incidents data ───

EVENT_TYPES    = ["Breakdown", "Accident", "Delay", "Route Diversion", "Overcrowding"]
SEVERITIES     = ["Low", "Medium", "High", "Critical"]

events = [
    {
        "event_id"    : f"EV{str(i).zfill(5)}",
        "vehicle_id"  : f"VH{str(random.randint(1, 200)).zfill(4)}",
        "route_id"    : f"RT{str(random.randint(1, 50)).zfill(3)}",
        "event_type"  : random.choice(EVENT_TYPES),
        "severity"    : random.choice(SEVERITIES),
        "event_ts"    : (base_ts + timedelta(
                            days=random.randint(0, 364),
                            hours=random.randint(0, 23)
                        )).isoformat(),
        "resolved"    : random.choice([True, True, False])
    }
    for i in range(1, 501)
]

events_df = spark.createDataFrame(events)
events_df.write.mode("overwrite").json(f"{LANDING_PATH}/events")

print(f"✅ Generated {events_df.count()} event records")
events_df.show(5, truncate=False)

---
## 🥉 Section 3: Bronze Layer — Raw Ingestion

In Lakeflow Declarative Pipelines, Bronze tables ingest raw data **as-is** using `@dlt.table` decorators and **Auto Loader** (`cloudFiles`).  
Below we show both:
1. The **Lakeflow pipeline syntax** (how you'd write it in a Databricks DLT/Lakeflow notebook)
2. The **equivalent PySpark simulation** for local runs

In [ ]:
# ════════════════════════════════════════════════════════
#  LAKEFLOW DECLARATIVE PIPELINE SYNTAX
#  (Run this cell inside a Databricks DLT Pipeline notebook)
# ════════════════════════════════════════════════════════

LAKEFLOW_BRONZE_CODE = '''
import dlt
from pyspark.sql import functions as F

# ── Bronze: Trips ──────────────────────────────────────
@dlt.table(
    name    = "bronze_trips",
    comment = "Raw trip records ingested via Auto Loader from landing zone",
    table_properties = {"quality": "bronze", "delta.enableChangeDataFeed": "true"}
)
def bronze_trips():
    return (
        spark.readStream
             .format("cloudFiles")                       # Auto Loader
             .option("cloudFiles.format", "json")
             .option("cloudFiles.schemaLocation", f"{CHECKPOINT}/trips_schema")
             .load(f"{LANDING_PATH}/trips")
             .withColumn("_ingested_at", F.current_timestamp())
             .withColumn("_source_file", F.input_file_name())
    )


# ── Bronze: Vehicles ───────────────────────────────────
@dlt.table(
    name    = "bronze_vehicles",
    comment = "Raw vehicle master data",
    table_properties = {"quality": "bronze"}
)
def bronze_vehicles():
    return (
        spark.readStream
             .format("cloudFiles")
             .option("cloudFiles.format", "json")
             .option("cloudFiles.schemaLocation", f"{CHECKPOINT}/vehicles_schema")
             .load(f"{LANDING_PATH}/vehicles")
             .withColumn("_ingested_at", F.current_timestamp())
    )


# ── Bronze: Routes ─────────────────────────────────────
@dlt.table(
    name    = "bronze_routes",
    comment = "Raw route definitions",
    table_properties = {"quality": "bronze"}
)
def bronze_routes():
    return (
        spark.readStream
             .format("cloudFiles")
             .option("cloudFiles.format", "json")
             .option("cloudFiles.schemaLocation", f"{CHECKPOINT}/routes_schema")
             .load(f"{LANDING_PATH}/routes")
             .withColumn("_ingested_at", F.current_timestamp())
    )


# ── Bronze: Events ─────────────────────────────────────
@dlt.table(
    name    = "bronze_events",
    comment = "Raw incident and event stream",
    table_properties = {"quality": "bronze"}
)
def bronze_events():
    return (
        spark.readStream
             .format("cloudFiles")
             .option("cloudFiles.format", "json")
             .option("cloudFiles.schemaLocation", f"{CHECKPOINT}/events_schema")
             .load(f"{LANDING_PATH}/events")
             .withColumn("_ingested_at", F.current_timestamp())
    )
'''

print("📋 Lakeflow Bronze Pipeline Definition (copy this into a DLT notebook):")
print(LAKEFLOW_BRONZE_CODE)

In [ ]:
# ─── PySpark Simulation: Bronze Layer ───
#  Equivalent batch read for local/notebook execution

bronze_trips_df = (
    spark.read.json(f"{LANDING_PATH}/trips")
         .withColumn("_ingested_at", F.current_timestamp())
         .withColumn("_source", F.lit("landing/trips"))
)

bronze_vehicles_df = (
    spark.read.json(f"{LANDING_PATH}/vehicles")
         .withColumn("_ingested_at", F.current_timestamp())
)

bronze_routes_df = (
    spark.read.json(f"{LANDING_PATH}/routes")
         .withColumn("_ingested_at", F.current_timestamp())
)

bronze_events_df = (
    spark.read.json(f"{LANDING_PATH}/events")
         .withColumn("_ingested_at", F.current_timestamp())
)

# Persist to Delta
bronze_trips_df.write.format("delta").mode("overwrite").save(f"{BRONZE_PATH}/trips")
bronze_vehicles_df.write.format("delta").mode("overwrite").save(f"{BRONZE_PATH}/vehicles")
bronze_routes_df.write.format("delta").mode("overwrite").save(f"{BRONZE_PATH}/routes")
bronze_events_df.write.format("delta").mode("overwrite").save(f"{BRONZE_PATH}/events")

print("✅ Bronze Layer written to Delta")
print(f"   ├─ Trips    : {bronze_trips_df.count():>6,} rows")
print(f"   ├─ Vehicles : {bronze_vehicles_df.count():>6,} rows")
print(f"   ├─ Routes   : {bronze_routes_df.count():>6,} rows")
print(f"   └─ Events   : {bronze_events_df.count():>6,} rows")

---
## 🥈 Section 4: Silver Layer — Cleansing, Validation & Enrichment

Silver layer applies:
- **Schema enforcement** and type casting
- **Data quality expectations** (`@dlt.expect` / `@dlt.expect_or_drop`)
- **Null handling** and deduplication
- **Joins** (trips ↔ vehicles ↔ routes)

In [ ]:
# ════════════════════════════════════════════════════════
#  LAKEFLOW DECLARATIVE PIPELINE SYNTAX — Silver Layer
# ════════════════════════════════════════════════════════

LAKEFLOW_SILVER_CODE = '''
import dlt
from pyspark.sql import functions as F

# ── Silver: Vehicles (cleaned) ─────────────────────────
@dlt.table(
    name    = "silver_vehicles",
    comment = "Cleaned and validated vehicle master data",
    table_properties = {"quality": "silver"}
)
@dlt.expect("valid_vehicle_id",   "vehicle_id IS NOT NULL")
@dlt.expect("valid_capacity",     "capacity > 0")
@dlt.expect_or_drop("active_vehicle", "is_active = true")   # Drop inactive vehicles
def silver_vehicles():
    return (
        dlt.read("bronze_vehicles")
           .select(
               F.col("vehicle_id"),
               F.col("vehicle_type"),
               F.col("capacity").cast("integer"),
               F.col("fuel_type"),
               F.col("city"),
               F.col("manufacture_year").cast("integer"),
               F.col("is_active").cast("boolean")
           )
           .dropDuplicates(["vehicle_id"])
    )


# ── Silver: Routes (cleaned) ───────────────────────────
@dlt.table(
    name    = "silver_routes",
    comment = "Cleaned route master data",
    table_properties = {"quality": "silver"}
)
@dlt.expect("valid_route_id",    "route_id IS NOT NULL")
@dlt.expect("valid_distance",    "distance_km > 0")
def silver_routes():
    return (
        dlt.read("bronze_routes")
           .select(
               F.col("route_id"),
               F.col("route_name"),
               F.col("origin_zone"),
               F.col("destination_zone"),
               F.col("distance_km").cast("double"),
               F.col("expected_duration_min").cast("integer"),
               F.col("city")
           )
           .dropDuplicates(["route_id"])
    )


# ── Silver: Trips (cleaned + enriched) ────────────────
@dlt.table(
    name    = "silver_trips",
    comment = "Cleaned trips joined with vehicle and route metadata",
    table_properties = {"quality": "silver", "delta.enableChangeDataFeed": "true"}
)
@dlt.expect("valid_trip_id",         "trip_id IS NOT NULL")
@dlt.expect("valid_fare",            "fare_amount > 0")
@dlt.expect_or_drop("valid_status",  "status IN (\'completed\', \'cancelled\', \'in_progress\')") 
@dlt.expect("non_negative_delay",    "delay_min >= 0")
def silver_trips():
    trips    = dlt.read("bronze_trips")
    vehicles = dlt.read("silver_vehicles")
    routes   = dlt.read("silver_routes")

    return (
        trips
        .select(
            F.col("trip_id"),
            F.col("vehicle_id"),
            F.col("route_id"),
            F.col("passenger_count").cast("integer"),
            F.to_timestamp(F.col("trip_start_ts")).alias("trip_start_ts"),
            F.to_timestamp(F.col("trip_end_ts")).alias("trip_end_ts"),
            F.col("duration_min").cast("integer"),
            F.col("fare_amount").cast("double"),
            F.col("payment_mode"),
            F.col("status"),
            F.col("delay_min").cast("integer")
        )
        .withColumn("trip_date",   F.to_date("trip_start_ts"))
        .withColumn("trip_hour",   F.hour("trip_start_ts"))
        .withColumn("trip_month",  F.month("trip_start_ts"))
        .withColumn("is_delayed",  F.col("delay_min") > 0)
        .join(vehicles.select("vehicle_id", "vehicle_type", "fuel_type", "city"),
              on="vehicle_id", how="left")
        .join(routes.select("route_id", "route_name", "origin_zone",
                             "destination_zone", "distance_km", "expected_duration_min"),
              on="route_id", how="left")
        .dropDuplicates(["trip_id"])
    )


# ── Silver: Events (cleaned) ───────────────────────────
@dlt.table(
    name    = "silver_events",
    comment = "Cleaned incident and event records",
    table_properties = {"quality": "silver"}
)
@dlt.expect("valid_event_id", "event_id IS NOT NULL")
def silver_events():
    return (
        dlt.read("bronze_events")
           .select(
               F.col("event_id"),
               F.col("vehicle_id"),
               F.col("route_id"),
               F.col("event_type"),
               F.col("severity"),
               F.to_timestamp(F.col("event_ts")).alias("event_ts"),
               F.col("resolved").cast("boolean")
           )
           .withColumn("event_date", F.to_date("event_ts"))
           .dropDuplicates(["event_id"])
    )
'''

print("📋 Lakeflow Silver Pipeline Definition:")
print(LAKEFLOW_SILVER_CODE)

In [ ]:
# ─── PySpark Simulation: Silver Layer ───

# Read from Bronze Delta tables
b_trips    = spark.read.format("delta").load(f"{BRONZE_PATH}/trips")
b_vehicles = spark.read.format("delta").load(f"{BRONZE_PATH}/vehicles")
b_routes   = spark.read.format("delta").load(f"{BRONZE_PATH}/routes")
b_events   = spark.read.format("delta").load(f"{BRONZE_PATH}/events")

# ── Silver: Vehicles ──
silver_vehicles_df = (
    b_vehicles
    .filter(F.col("is_active") == True)
    .select(
        F.col("vehicle_id"),
        F.col("vehicle_type"),
        F.col("capacity").cast("integer"),
        F.col("fuel_type"),
        F.col("city"),
        F.col("manufacture_year").cast("integer"),
        F.col("is_active").cast("boolean")
    )
    .dropDuplicates(["vehicle_id"])
    .filter(F.col("vehicle_id").isNotNull())
)

# ── Silver: Routes ──
silver_routes_df = (
    b_routes
    .select(
        "route_id", "route_name", "origin_zone", "destination_zone",
        F.col("distance_km").cast("double"),
        F.col("expected_duration_min").cast("integer"),
        "city"
    )
    .dropDuplicates(["route_id"])
    .filter(F.col("route_id").isNotNull())
)

# ── Silver: Trips (with joins) ──
silver_trips_df = (
    b_trips
    .select(
        "trip_id", "vehicle_id", "route_id",
        F.col("passenger_count").cast("integer"),
        F.to_timestamp("trip_start_ts").alias("trip_start_ts"),
        F.to_timestamp("trip_end_ts").alias("trip_end_ts"),
        F.col("duration_min").cast("integer"),
        F.col("fare_amount").cast("double"),
        "payment_mode", "status",
        F.col("delay_min").cast("integer")
    )
    .filter(F.col("status").isin("completed", "cancelled", "in_progress"))
    .filter(F.col("fare_amount") > 0)
    .withColumn("trip_date",  F.to_date("trip_start_ts"))
    .withColumn("trip_hour",  F.hour("trip_start_ts"))
    .withColumn("trip_month", F.month("trip_start_ts"))
    .withColumn("is_delayed", F.col("delay_min") > 0)
    .join(
        silver_vehicles_df.select("vehicle_id", "vehicle_type", "fuel_type", "city"),
        on="vehicle_id", how="left"
    )
    .join(
        silver_routes_df.select("route_id", "route_name", "origin_zone",
                                 "destination_zone", "distance_km", "expected_duration_min"),
        on="route_id", how="left"
    )
    .dropDuplicates(["trip_id"])
)

# ── Silver: Events ──
silver_events_df = (
    b_events
    .select(
        "event_id", "vehicle_id", "route_id", "event_type", "severity",
        F.to_timestamp("event_ts").alias("event_ts"),
        F.col("resolved").cast("boolean")
    )
    .withColumn("event_date", F.to_date("event_ts"))
    .dropDuplicates(["event_id"])
)

# Persist Silver
silver_trips_df.write.format("delta").mode("overwrite").save(f"{SILVER_PATH}/trips")
silver_vehicles_df.write.format("delta").mode("overwrite").save(f"{SILVER_PATH}/vehicles")
silver_routes_df.write.format("delta").mode("overwrite").save(f"{SILVER_PATH}/routes")
silver_events_df.write.format("delta").mode("overwrite").save(f"{SILVER_PATH}/events")

print("✅ Silver Layer written to Delta")
print(f"   ├─ Trips    : {silver_trips_df.count():>6,} rows (after DQ filters)")
print(f"   ├─ Vehicles : {silver_vehicles_df.count():>6,} rows (active only)")
print(f"   ├─ Routes   : {silver_routes_df.count():>6,} rows")
print(f"   └─ Events   : {silver_events_df.count():>6,} rows")
silver_trips_df.printSchema()

---
## 🥇 Section 5: Gold Layer — Business KPIs & Aggregations

Gold tables are business-ready, aggregated tables used by dashboards, BI tools, and analysts.

In [ ]:
# ════════════════════════════════════════════════════════
#  LAKEFLOW DECLARATIVE PIPELINE SYNTAX — Gold Layer
# ════════════════════════════════════════════════════════

LAKEFLOW_GOLD_CODE = '''
import dlt
from pyspark.sql import functions as F

# ── Gold: Daily Trip Summary ───────────────────────────
@dlt.table(
    name    = "gold_daily_trip_summary",
    comment = "Daily KPIs: trips, revenue, passengers, delays by city and vehicle type",
    table_properties = {"quality": "gold"}
)
def gold_daily_trip_summary():
    return (
        dlt.read("silver_trips")
           .filter(F.col("status") == "completed")
           .groupBy("trip_date", "city", "vehicle_type", "trip_month")
           .agg(
               F.count("trip_id")                      .alias("total_trips"),
               F.sum("fare_amount")                    .alias("total_revenue"),
               F.avg("fare_amount")                    .alias("avg_fare"),
               F.sum("passenger_count")                .alias("total_passengers"),
               F.avg("duration_min")                   .alias("avg_duration_min"),
               F.avg("delay_min")                      .alias("avg_delay_min"),
               F.sum(F.col("is_delayed").cast("int"))  .alias("delayed_trips"),
               F.countDistinct("vehicle_id")           .alias("unique_vehicles")
           )
           .withColumn("delay_rate_pct",
               F.round(F.col("delayed_trips") / F.col("total_trips") * 100, 2))
           .withColumn("revenue_per_trip",
               F.round(F.col("total_revenue") / F.col("total_trips"), 2))
    )


# ── Gold: Route Performance ────────────────────────────
@dlt.table(
    name    = "gold_route_performance",
    comment = "Route-level performance: on-time %, revenue, utilization",
    table_properties = {"quality": "gold"}
)
def gold_route_performance():
    return (
        dlt.read("silver_trips")
           .filter(F.col("status") == "completed")
           .groupBy("route_id", "route_name", "origin_zone", "destination_zone", "distance_km")
           .agg(
               F.count("trip_id")              .alias("total_trips"),
               F.sum("fare_amount")            .alias("total_revenue"),
               F.avg("delay_min")              .alias("avg_delay_min"),
               F.avg("passenger_count")        .alias("avg_passengers"),
               F.avg("duration_min")           .alias("avg_actual_duration_min"),
               F.first("expected_duration_min").alias("expected_duration_min")
           )
           .withColumn("efficiency_ratio",
               F.round(F.col("expected_duration_min") / F.col("avg_actual_duration_min"), 3))
           .withColumn("revenue_per_km",
               F.round(F.col("total_revenue") / F.col("distance_km"), 2))
           .orderBy(F.col("total_revenue").desc())
    )


# ── Gold: Vehicle Utilization ──────────────────────────
@dlt.table(
    name    = "gold_vehicle_utilization",
    comment = "Vehicle-level operational metrics and fuel type analysis",
    table_properties = {"quality": "gold"}
)
def gold_vehicle_utilization():
    return (
        dlt.read("silver_trips")
           .groupBy("vehicle_id", "vehicle_type", "fuel_type", "city")
           .agg(
               F.count("trip_id")                      .alias("total_trips"),
               F.sum("fare_amount")                    .alias("total_revenue"),
               F.sum("passenger_count")                .alias("total_passengers_carried"),
               F.avg("delay_min")                      .alias("avg_delay_min"),
               F.sum(F.col("is_delayed").cast("int"))  .alias("delayed_trips"),
               F.countDistinct("route_id")             .alias("unique_routes_served")
           )
           .withColumn("reliability_score",
               F.round(
                   (1 - F.col("delayed_trips") / F.col("total_trips")) * 100, 2
               )
           )
    )


# ── Gold: Incident Impact Analysis ────────────────────
@dlt.table(
    name    = "gold_incident_impact",
    comment = "Event/incident analysis by type, severity, and resolution status",
    table_properties = {"quality": "gold"}
)
def gold_incident_impact():
    return (
        dlt.read("silver_events")
           .groupBy("event_type", "severity", "resolved")
           .agg(
               F.count("event_id")          .alias("total_incidents"),
               F.countDistinct("vehicle_id").alias("vehicles_affected"),
               F.countDistinct("route_id")  .alias("routes_affected")
           )
           .orderBy("severity", F.col("total_incidents").desc())
    )
'''

print("📋 Lakeflow Gold Pipeline Definition:")
print(LAKEFLOW_GOLD_CODE)

In [ ]:
# ─── PySpark Simulation: Gold Layer ───

s_trips  = spark.read.format("delta").load(f"{SILVER_PATH}/trips")
s_events = spark.read.format("delta").load(f"{SILVER_PATH}/events")

# ── Gold 1: Daily Trip Summary ──
gold_daily_df = (
    s_trips
    .filter(F.col("status") == "completed")
    .groupBy("trip_date", "city", "vehicle_type", "trip_month")
    .agg(
        F.count("trip_id")                     .alias("total_trips"),
        F.round(F.sum("fare_amount"), 2)       .alias("total_revenue"),
        F.round(F.avg("fare_amount"), 2)       .alias("avg_fare"),
        F.sum("passenger_count")               .alias("total_passengers"),
        F.round(F.avg("duration_min"), 2)      .alias("avg_duration_min"),
        F.round(F.avg("delay_min"), 2)         .alias("avg_delay_min"),
        F.sum(F.col("is_delayed").cast("int")) .alias("delayed_trips"),
        F.countDistinct("vehicle_id")          .alias("unique_vehicles")
    )
    .withColumn("delay_rate_pct",
        F.round(F.col("delayed_trips") / F.col("total_trips") * 100, 2))
    .withColumn("revenue_per_trip",
        F.round(F.col("total_revenue") / F.col("total_trips"), 2))
)

# ── Gold 2: Route Performance ──
gold_routes_df = (
    s_trips
    .filter(F.col("status") == "completed")
    .groupBy("route_id", "route_name", "origin_zone", "destination_zone", "distance_km")
    .agg(
        F.count("trip_id")              .alias("total_trips"),
        F.round(F.sum("fare_amount"), 2).alias("total_revenue"),
        F.round(F.avg("delay_min"), 2)  .alias("avg_delay_min"),
        F.round(F.avg("passenger_count"), 1).alias("avg_passengers"),
        F.round(F.avg("duration_min"), 2)  .alias("avg_actual_duration_min"),
        F.first("expected_duration_min")   .alias("expected_duration_min")
    )
    .withColumn("efficiency_ratio",
        F.round(F.col("expected_duration_min") / F.col("avg_actual_duration_min"), 3))
    .withColumn("revenue_per_km",
        F.round(F.col("total_revenue") / F.col("distance_km"), 2))
    .orderBy(F.col("total_revenue").desc())
)

# ── Gold 3: Vehicle Utilization ──
gold_vehicles_df = (
    s_trips
    .groupBy("vehicle_id", "vehicle_type", "fuel_type", "city")
    .agg(
        F.count("trip_id")                     .alias("total_trips"),
        F.round(F.sum("fare_amount"), 2)       .alias("total_revenue"),
        F.sum("passenger_count")               .alias("total_passengers_carried"),
        F.round(F.avg("delay_min"), 2)         .alias("avg_delay_min"),
        F.sum(F.col("is_delayed").cast("int")) .alias("delayed_trips"),
        F.countDistinct("route_id")            .alias("unique_routes_served")
    )
    .withColumn("reliability_score",
        F.round((1 - F.col("delayed_trips") / F.col("total_trips")) * 100, 2))
)

# ── Gold 4: Incident Impact ──
gold_incidents_df = (
    s_events
    .groupBy("event_type", "severity", "resolved")
    .agg(
        F.count("event_id")          .alias("total_incidents"),
        F.countDistinct("vehicle_id").alias("vehicles_affected"),
        F.countDistinct("route_id")  .alias("routes_affected")
    )
    .orderBy("severity", F.col("total_incidents").desc())
)

# Persist Gold
gold_daily_df.write.format("delta").mode("overwrite").save(f"{GOLD_PATH}/daily_trip_summary")
gold_routes_df.write.format("delta").mode("overwrite").save(f"{GOLD_PATH}/route_performance")
gold_vehicles_df.write.format("delta").mode("overwrite").save(f"{GOLD_PATH}/vehicle_utilization")
gold_incidents_df.write.format("delta").mode("overwrite").save(f"{GOLD_PATH}/incident_impact")

print("✅ Gold Layer written to Delta")
print(f"   ├─ Daily Summary      : {gold_daily_df.count():>5,} rows")
print(f"   ├─ Route Performance  : {gold_routes_df.count():>5,} rows")
print(f"   ├─ Vehicle Utilization: {gold_vehicles_df.count():>5,} rows")
print(f"   └─ Incident Impact    : {gold_incidents_df.count():>5,} rows")

---
## 📊 Section 6: Business Insights & Analysis

In [ ]:
# ─── KPI 1: Monthly Revenue by City ───

print("═" * 60)
print("  📈  Monthly Revenue by City")
print("═" * 60)

gold_daily_df.groupBy("city", "trip_month") \
    .agg(F.round(F.sum("total_revenue"), 2).alias("monthly_revenue"),
         F.sum("total_trips").alias("monthly_trips")) \
    .orderBy("city", "trip_month") \
    .show(20, truncate=False)

In [ ]:
# ─── KPI 2: Top 10 Routes by Revenue ───

print("═" * 70)
print("  🏆  Top 10 Routes by Revenue")
print("═" * 70)

gold_routes_df.select(
    "route_name", "origin_zone", "destination_zone",
    "total_trips", "total_revenue", "revenue_per_km",
    "avg_delay_min", "efficiency_ratio"
).limit(10).show(truncate=False)

In [ ]:
# ─── KPI 3: Vehicle Reliability by Fuel Type ───

print("═" * 60)
print("  🔋  Reliability Score by Fuel Type")
print("═" * 60)

gold_vehicles_df \
    .groupBy("fuel_type") \
    .agg(
        F.round(F.avg("reliability_score"), 2).alias("avg_reliability_score"),
        F.round(F.avg("avg_delay_min"), 2)    .alias("avg_delay_min"),
        F.sum("total_trips")                  .alias("total_trips")
    ) \
    .orderBy(F.col("avg_reliability_score").desc()) \
    .show(truncate=False)

In [ ]:
# ─── KPI 4: Incident Severity Breakdown ───

print("═" * 60)
print("  🚨  Incident Severity Breakdown")
print("═" * 60)

gold_incidents_df.show(20, truncate=False)

In [ ]:
# ─── KPI 5: Peak Hour Analysis ───

print("═" * 60)
print("  🕐  Peak Hour Trip Distribution")
print("═" * 60)

s_trips.filter(F.col("status") == "completed") \
    .groupBy("trip_hour") \
    .agg(
        F.count("trip_id")                     .alias("total_trips"),
        F.round(F.avg("fare_amount"), 2)       .alias("avg_fare"),
        F.round(F.avg("delay_min"), 2)         .alias("avg_delay_min")
    ) \
    .orderBy("trip_hour") \
    .show(24, truncate=False)

---
## 🔍 Section 7: Data Quality Monitoring

In Lakeflow pipelines, expectations (DQ rules) are tracked automatically. Here we simulate a DQ report.

In [ ]:
# ─── Data Quality Report ───

def dq_check(df, name, checks: dict):
    """Run data quality checks and print a summary report."""
    total = df.count()
    print(f"\n{'═'*55}")
    print(f"  DQ Report: {name}  (total rows: {total:,})")
    print(f"{'═'*55}")
    for rule_name, condition in checks.items():
        passed = df.filter(condition).count()
        failed = total - passed
        pct    = round(passed / total * 100, 2) if total > 0 else 0
        status = "✅" if failed == 0 else "⚠️ "
        print(f"  {status} {rule_name:<30} passed: {passed:>5,}  failed: {failed:>4,}  ({pct}%)")

dq_check(s_trips, "silver_trips", {
    "trip_id not null"         : F.col("trip_id").isNotNull(),
    "fare > 0"                 : F.col("fare_amount") > 0,
    "passenger_count > 0"      : F.col("passenger_count") > 0,
    "valid status"             : F.col("status").isin("completed","cancelled","in_progress"),
    "delay_min non-negative"   : F.col("delay_min") >= 0,
    "trip_start before end"    : F.col("trip_start_ts") < F.col("trip_end_ts")
})

dq_check(silver_vehicles_df, "silver_vehicles", {
    "vehicle_id not null"      : F.col("vehicle_id").isNotNull(),
    "capacity > 0"             : F.col("capacity") > 0,
    "manufacture_year valid"   : F.col("manufacture_year").between(2000, 2024)
})

---
## 🏁 Section 8: Pipeline Summary

### What We Built

| Layer  | Tables | Description |
|--------|--------|-------------|
| Bronze | 4      | Raw ingestion via Auto Loader (trips, vehicles, routes, events) |
| Silver | 4      | Cleaned, validated, joined with DQ expectations |
| Gold   | 4      | Business KPIs (daily summary, route perf, vehicle util, incidents) |

### Key Lakeflow Concepts Demonstrated
- `@dlt.table` — declarative table definitions
- `@dlt.expect` / `@dlt.expect_or_drop` — data quality expectations
- `cloudFiles` (Auto Loader) — incremental, scalable file ingestion
- `dlt.read()` — dependency-aware table references in the pipeline DAG
- Medallion architecture — Bronze → Silver → Gold
- Delta Lake — ACID transactions, time travel, schema evolution
- Unity Catalog integration — governed, discoverable data assets

### Technologies
- **Databricks** (Lakeflow / DLT)
- **Apache Spark** (PySpark)
- **Delta Lake**
- **Unity Catalog**
- **Auto Loader** (cloudFiles)

---
*Built as part of the Databricks Lakeflow Spark Declarative Pipelines micro-course.*